# Ollama GPU Host Server on Kaggle (Cloudflare Tunnel)

This notebook runs on Kaggle / Google Colab

### 📋 Instructions for Kaggle:
1. Make sure to set **Accelerator = GPU T4 x2** and **Internet = On**.
2. Wait a few minutes for the model to download, then copy the HTTPS `trycloudflare.com` URL from the printed output and paste it into your local `.env` file.

In [ ]:
import os
import re
import subprocess
import time


# ---------------------------------------------------------
# 1. Install required libraries and Ollama
# ---------------------------------------------------------
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
# ---------------------------------------------------------
# 2. Install Cloudflared
# ---------------------------------------------------------
if not os.path.exists("cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared", shell=True, check=True)
    subprocess.run("chmod +x cloudflared", shell=True, check=True)

# ---------------------------------------------------------
# 3. Start Ollama Server with OLLAMA_ORIGINS=* (Fix 403 CORS error)
# ---------------------------------------------------------
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"] = "0.0.0.0:11434"
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)

# ---------------------------------------------------------
# 4. Download LLM Model (qwen2.5:3b)
# ---------------------------------------------------------
MODEL_NAME = "qwen2.5:3b"
subprocess.run(["ollama", "pull", MODEL_NAME], check=True, env=env)

# ---------------------------------------------------------
# 5. Create Cloudflare Tunnel to Port 11434
# ---------------------------------------------------------
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

for line in tunnel_proc.stdout:
    match = pattern.search(line)
    if match:
        public_url = match.group(0)
        break

print("SUCCESS! OLLAMA SERVER IS READY ON KAGGLE GPU!")
print(f" YOUR PUBLIC URL IS: {public_url}")
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("Server stopped.")
